# Social Media Sentiment Analysis on a Trending Topic

**Project:** Airline Customer Experience & Service Conversations  
**Tool:** Python — Pandas, NLTK/VADER, Matplotlib, WordCloud

> **Dataset scope:** The supplied Twitter Airline Sentiment dataset is historical (February 2015). It demonstrates the required workflow but should not be presented as current 2026 public opinion.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

df = pd.read_csv('../01_data/airline_sentiment_cleaned.csv')
try:
    sia = SentimentIntensityAnalyzer()
except LookupError:
    nltk.download('vader_lexicon')
    sia = SentimentIntensityAnalyzer()
df.head()

## 1. Dataset overview

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('\nSentiment counts:')
print(df['airline_sentiment'].value_counts())
print('\nAirlines:')
print(df['airline'].value_counts())

## 2. Sentiment distribution

In [ ]:
order = ['positive','neutral','negative']
df['airline_sentiment'].value_counts().reindex(order).plot(kind='bar')
plt.title('Overall Sentiment Distribution')
plt.xlabel('Sentiment'); plt.ylabel('Tweet Count')
plt.tight_layout(); plt.show()

## 3. Sentiment by airline

In [ ]:
cross = pd.crosstab(df['airline'], df['airline_sentiment']).reindex(columns=order, fill_value=0)
cross.plot(kind='bar', stacked=True)
plt.title('Sentiment by Airline')
plt.xlabel('Airline'); plt.ylabel('Tweet Count')
plt.xticks(rotation=35); plt.tight_layout(); plt.show()

## 4. Sentiment over time

In [ ]:
df['date'] = pd.to_datetime(df['date'])
daily = pd.crosstab(df['date'], df['airline_sentiment']).reindex(columns=order, fill_value=0)
daily_pct = daily.div(daily.sum(axis=1), axis=0) * 100
daily_pct.plot()
plt.title('Sentiment Share Over Time')
plt.xlabel('Date'); plt.ylabel('Share (%)')
plt.tight_layout(); plt.show()

## 5. VADER NLP validation

In [ ]:
df['vader_compound'] = df['text'].fillna('').map(lambda x: sia.polarity_scores(x)['compound'])
df['vader_sentiment'] = df['vader_compound'].apply(lambda x: 'positive' if x >= 0.05 else ('negative' if x <= -0.05 else 'neutral'))
agreement = (df['vader_sentiment'] == df['airline_sentiment']).mean() * 100
print(f'VADER agreement with provided labels: {agreement:.2f}%')

## 6. Negative reasons

In [ ]:
reasons = df.loc[df['airline_sentiment'].eq('negative'), 'negativereason'].value_counts().head(10)
reasons.sort_values().plot(kind='barh')
plt.title('Top Negative Reasons'); plt.xlabel('Tweet Count')
plt.tight_layout(); plt.show()

## 7. Word cloud

In [ ]:
negative_text = ' '.join(df.loc[df['airline_sentiment'].eq('negative'), 'text_clean'].astype(str))
wc = WordCloud(width=1400, height=800, background_color='white', collocations=False).generate(negative_text)
plt.figure(figsize=(14,8)); plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
plt.title('Negative Tweet Keyword Word Cloud'); plt.show()

## 8. Interpretation

Discuss the overall sentiment mix, differences among airlines, time patterns, the main negative reasons, and recurring keywords. Keep conclusions limited to the February 2015 dataset period.